#### Validation & Quarantine Layer

###Create quarantine table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS finance_dev.quarantine_transactions (
    TransactionID STRING,
    AccountID STRING,
    TransactionDate DATE,
    Amount DOUBLE,
    Currency STRING,
    TransactionType STRING,
    CostCenter STRING,
    BusinessUnit STRING,
    IngestionTime TIMESTAMP,
    SourceFile STRING,
    RunID STRING,
    ErrorReason STRING,
    QuarantinedAt TIMESTAMP
) USING DELTA;

###Read Bronze tables

In [0]:
from pyspark.sql.functions import *
tx=spark.table("finance_dev.bronze_transactions")
accounts=spark.table("finance_dev.bronze_accounts")
rates=spark.table("finance_dev.bronze_exchange_rates")

display(tx.count())
display(accounts.count())
display(rates.count())

###Validation 1: NULL Amount

In [0]:
null_amount=tx.filter(col("Amount").isNull()).withColumn("ErrorReason",lit("NULL Amount")).withColumn("QuarantinedAt",current_timestamp())
display(null_amount)

###Validation 2: Invalid Currency - referential integrity

In [0]:
invalid_currency=tx.join(rates,rates.Currency==tx.Currency,"leftanti").withColumn("ErrorReason",lit("Invalid Currency")).withColumn("QuarantinedAt",current_timestamp())
display(invalid_currency)

###Validation 3: Invalid accounts - Referential _Integrity

In [0]:
invalid_accounts=tx.join(accounts,accounts.AccountID==tx.AccountID,"leftanti").withColumn("ErrorReason",lit("Invalid Accounts")).withColumn("QuarantinedAt",current_timestamp())
display(invalid_accounts)

###Validation 4: Future Dates

In [0]:
invalid_dates=tx.filter(col("TransactionDate")>current_date()).withColumn("ErrorReason",lit("Future Date")).withColumn("QuarantinedAt",current_timestamp())
display(invalid_dates)

###Validation 5: Duplicate TransactionID

In [0]:
from pyspark.sql.window import Window

w=Window.partitionBy("TransactionID").orderBy(col("IngestionTime").desc())

Dup_transactions=tx.withColumn("rn",row_number().over(w))\
    .filter(col("rn")>1)\
    .drop("rn")\
    .withColumn("ErrorReason", lit("DUPLICATE_TRANSACTION_ID"))\
    .withColumn("QuarantinedAt", current_timestamp())\

display(Dup_transactions)

###Combine all bad records

In [0]:
quarantine_df=null_amount.unionByName(invalid_currency).unionByName(invalid_accounts).unionByName(invalid_dates).unionByName(Dup_transactions)
display(quarantine_df)

In [0]:
(quarantine_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("finance_dev.quarantine_transactions"))

###Save valid transaction records into silver layer

In [0]:
invalid_ids=quarantine_df.select("TransactionID").distinct()

valid_records=tx.join(invalid_ids,tx.TransactionID==invalid_ids.TransactionID,"leftanti")

print("Valid records:", valid_records.count())
print("Quarantined records:", quarantine_df.count())

(valid_records.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("finance_dev.silver_transactions_staging"))

###DQ Metrics

In [0]:
dq_metrics = spark.createDataFrame([
    ("TOTAL_RECORDS", tx.count()),
    ("NULL_AMOUNT", null_amount.count()),
    ("INVALID_CURRENCY", invalid_currency.count()),
    ("INVALID_ACCOUNT", invalid_accounts.count()),
    ("FUTURE_DATE", invalid_dates.count()),
    ("DUPLICATE_TRANSACTION_ID", Dup_transactions.count()),
    ("VALID_RECORDS", valid_records.count())
], ["Metric", "Count"])

display(dq_metrics)